In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
# 1. Load the breast cancer dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 = malignant, 1 = benign
# 2. Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
# 3. Train AdaBoost with decision stumps
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    learning_rate=0.5,
    algorithm='SAMME',
    random_state=1
)
ada.fit(X_train, y_train)
# 4. Evaluate on test data
y_pred = ada.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("\n=== AdaBoost Classification Report ===")
print("Accuracy:", round(acc, 4))
print(classification_report(y_test, y_pred, target_names=data.target_names))
# 5. Predict on a new sample (first sample in test set)
sample = X_test.iloc[0:1]
pred = ada.predict(sample)[0]
prob = ada.predict_proba(sample)[0][pred]
label = data.target_names[pred]
print("\n=== New Sample Prediction ===")
print("Sample features:", sample.to_dict(orient='records')[0])
print(f"Predicted class: {label} with probability {prob:.3f}")
# 6. Optional: Plot feature importances
importances = ada.feature_importances_
feat_names = X.columns
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp[:10], y=feat_imp.index[:10], palette="viridis")
plt.title("Top 10 Feature Importances - AdaBoost")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()
''' Output -
=== AdaBoost Classification Report ===
Accuracy: 0.9561
              precision    recall  f1-score   support
   malignant       0.97      0.90      0.94        42
      benign       0.95      0.99      0.97        72
    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Predicted class: malignant with probability 0.834
'''